In [37]:
import json
import os
import pandas as pd
from PIL import Image

input_json   = "heico_2dbb_video.json"
path_to_images = "/home/j566n/SAVEOR_Yolo/FO_frames/"
output_images = "/home/j566n/SAVEOR_Yolo/HEICO_dataset/images/"
output_labels = "/home/j566n/SAVEOR_Yolo/HEICO_dataset/labels/"

output_csv   = "heico_2dbb_video.csv"

classes = ['Clip', 'SiliconLoop', 'Needle', 'SpecimenBag', 'Sponge', 'ExternalDrain']

In [51]:
with open(input_json, "r") as f:
    data = json.load(f)

print(f"Number of annotated images: {len(data)}")

df = pd.read_csv(output_csv)

print(f"Existing CSV has {len(df)} entries.")

for item in data:
    # print(f"Processing item: {item}")
    if item not in df['filename'].values:
        print(f"  Item {item} not in CSV, processing...")

Number of annotated images: 44717
Existing CSV has 44717 entries.


In [58]:
with open(input_json, "r") as f:
    data = json.load(f)

print(f"Number of annotated images: {len(data)}")
# file_names = os.listdir(output_images)
# # Convert to df
# df = pd.DataFrame(file_names, columns=['filename'])
# df.to_csv(output_csv, index=False)

df_rows = []

for filename, ann_list in data.items():
    file_dir = filename.split("_")[:2]
    file_dir = "_".join(file_dir)

    # if "Sigma_9_clip_3104_1" not in filename:
    #     continue

    # print(f"Processing file: {filename}, with {len(ann_list)} annotations and dir {file_dir}")

    if filename not in df['filename'].values:
        print(f"  Item {filename} not in CSV, processing...")
    
    if filename not in data:
        print(f"⚠️ Warning: No annotations found for image {filename}. Skipping.")
    continue
    
    img_path = os.path.join(path_to_images,file_dir, filename)

    # Load image size once per image
    with Image.open(img_path) as im:
        img_w, img_h = im.size

        # # Copy image to output directory
        # output_img_path = os.path.join(output_images, filename)
        # os.makedirs(os.path.dirname(output_img_path), exist_ok=True)
        # if not os.path.exists(output_img_path):
        #     im.save(output_img_path)

    # Write YOLO txt line
    yolo_txt_path = os.path.join(output_labels, filename.replace(".jpg", ".txt"))

    with open(yolo_txt_path, "w") as f:
        # Process all annotations for this image
        ann_classes = []
        ann_bboxes = []
        ann_instances = []
        ann_timepoints = []

        for ann in ann_list:
            cls_name = ann["class"]
            if cls_name not in classes:
                print(f"⚠️ Warning: class {cls_name} not in classes list. Skipping.")
                continue

            cls_id = classes.index(cls_name)
            x, y, w_box, h_box = ann["bbox"]  # x, y = top-left corner

            # Convert to YOLO format
            x_center = (x + w_box / 2) / img_w
            y_center = (y + h_box / 2) / img_h
            w_norm = w_box / img_w
            h_norm = h_box / img_h

            # print(f"    YOLO format: cls_id={cls_id}, x_center={x_center}, y_center={y_center}, w={w_norm}, h={h_norm}")
            f.write(f"{cls_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

            
            ann_classes.append(cls_name)
            ann_bboxes.append((x, y, w_box, h_box))
            ann_instances.append(ann.get("instance", None))
            ann_timepoints.append(ann.get("timepoint", None))
        
        print(f"ann_classes: {ann_classes}")
        print(f"ann_bboxes: {ann_bboxes}")
        print(f"ann_instances: {ann_instances}")
        print(f"ann_timepoints: {ann_timepoints}")
        # Add dataframe row
        df_rows.append({
            "filename": filename,
            "file_path": img_path,
            "classes": ann_classes,
            "bboxes": ann_bboxes,
            "instances": ann_instances,
            "timepoints": ann_timepoints
        })

# --------------------------------------------------------------------
# Save CSV
# --------------------------------------------------------------------
# df = pd.DataFrame(df_rows)
# df.to_csv(output_csv, index=False)

# print("✅ Conversion complete!")
# print(f"CSV written to: {output_csv}")
# print(f"YOLO TXT files saved to: {output_labels}")


Number of annotated images: 44717


## TRAIN TEST SPLIT

In [60]:
import os
import pandas as pd

df = pd.read_csv(output_csv)
path_to_testset = "/home/j566n/SAVEOR_Yolo/Curated_labels"

test_img_ids = [file_name.split('.')[0] for file_name in os.listdir(path_to_testset) if not "classes" in file_name]
print(f"Test image IDs: {test_img_ids} found.")

df['set'] = 'train'  # Default to train

for test_img in test_img_ids:
    if not any(df['filename'].str.contains(test_img)):
        print(f"⚠️ Warning: test image ID {test_img} not found in dataframe filenames.")

df.loc[df['filename'].str.replace('.jpg', '').isin(test_img_ids), 'set'] = 'test'

print(len(test_img_ids))
print(f"test set samples: {df[df['set']=='test'].shape[0]}")

# save updated csv
# df.to_csv(output_csv, index=False)

Test image IDs: ['Rektum_6_clip_0449_2', 'Rektum_1_clip_3168_0', 'Rektum_3_clip_0907_1', 'Prokto_3_clip_2180_2', 'Sigma_9_clip_0952_3', 'Rektum_3_clip_1721_3', 'Rektum_9_clip_2741_3', 'Sigma_8_clip_1578_3', 'Sigma_5_clip_0352_0', 'Prokto_3_clip_1908_2', 'Sigma_5_clip_0573_0', 'Rektum_4_clip_1777_2', 'Rektum_2_clip_2723_2', 'Prokto_7_clip_1705_3', 'Prokto_9_clip_2350_2', 'Rektum_9_clip_0933_2', 'Rektum_2_clip_2717_1', 'Prokto_4_clip_0065_3', 'Sigma_8_clip_1533_1', 'Rektum_2_clip_2599_0', 'Sigma_5_clip_0323_1', 'Rektum_1_clip_0673_0', 'Prokto_6_clip_1616_4', 'Rektum_8_clip_0562_2', 'Rektum_4_clip_0275_0', 'Prokto_4_clip_2071_1', 'Rektum_2_clip_2679_3', 'Rektum_2_clip_2726_1', 'Rektum_7_clip_2136_1', 'Prokto_4_clip_2068_0', 'Sigma_1_clip_0101_0', 'Sigma_5_clip_0487_4', 'Sigma_8_clip_0596_4', 'Prokto_8_clip_1759_1', 'Rektum_3_clip_1728_4', 'Prokto_7_clip_2029_4', 'Sigma_6_clip_2525_2', 'Prokto_3_clip_0463_3', 'Rektum_9_clip_2819_0', 'Sigma_5_clip_0493_3', 'Rektum_9_clip_0375_0', 'Rektum_9_

In [27]:
# Setup stratified CV for train images

from sklearn.model_selection import StratifiedKFold

df = pd.read_csv(output_csv)
df_train = df[df['set'] == 'train'].copy()

# Convert to actual Python list
df_train["classes"] = df_train["classes"].apply(eval)

# If multi-label, join with "+" so that each sample has ONE label
df_train["strat_label"] = df_train["classes"].apply(
    lambda x: x[0] if len(x) == 1 else "+".join(sorted(x))
)

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

df_train["fold"] = -1

for fold_id, (_, val_idx) in enumerate(
    skf.split(df_train, df_train["strat_label"])
):
    df_train.loc[df_train.index[val_idx], "fold"] = int(fold_id)

df = df.merge(df_train[["filename", "fold"]], on="filename", how="left")

# Save if needed
df.to_csv("heico_2dbb_video_with_folds.csv", index=False)

print(df["fold"].value_counts())
print("✅ Stratified CV assignment complete!")


fold
1.0    8817
0.0    8817
2.0    8816
4.0    8816
3.0    8816
Name: count, dtype: int64
✅ Stratified CV assignment complete!


/home/j566n/miniconda3/envs/cvs_clean/lib/python3.10/site-packages/sklearn/model_selection/_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
